<a href="https://colab.research.google.com/github/Mahid-Imran/flyrank-ml-internship-mahid-assignment2/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahid-Imran/flyrank-ml-internship-mahid-assignment2/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## My rule and reason codes

My baseline rule identifies pages that may benefit from content refresh.

The rule uses two signals:

1. Content staleness:
   - Pages that have not been updated for a long time receive a higher score.

2. Search opportunity:
   - Pages with higher impressions have more potential value if improved.

The final refresh score combines both signals.

Reason codes:

- STALE_HIGH_VALUE:
  Old page with strong search visibility.

- STALE_LOW_VALUE:
  Old page but with limited search opportunity.

- FRESH_HIGH_VALUE:
  Recently updated page with high visibility.

- LOW_PRIORITY:
  Page does not currently show strong refresh signals.

The output action labels are:

- REFRESH
- MONITOR
- NO_ACTION

In [1]:
import pandas as pd
import numpy as np
import os


# Load dataset
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "flyrank-bih/flyrank-ml-internship-starter/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)


print("Dataset shape:", df.shape)

df.head()

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:
# -----------------------------
# Signal 1: Staleness check
# -----------------------------

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0,180,365,10000],
    labels=[
        "Fresh",
        "Medium",
        "Stale"
    ]
)


staleness_check = (
    df.groupby("staleness_bucket",
               observed=False)
    ["trend_direction"]
    .value_counts(normalize=True)
    .unstack()
    .round(3)
)


print("Signal 1: Days since last update")
print(staleness_check)

Signal 1: Days since last update
trend_direction    down   flat    new  stable     up
staleness_bucket                                    
Fresh             0.542  0.038  0.074   0.199  0.146
Medium            0.467  0.089  0.142   0.142  0.160
Stale             0.600  0.200  0.200   0.000  0.000


In [3]:
# -----------------------------
# Signal 2: Search opportunity
# -----------------------------

df["impression_bucket"] = pd.qcut(
    df["impressions_90d"],
    q=4,
    labels=[
        "Low",
        "Medium Low",
        "Medium High",
        "High"
    ]
)


impression_check = (
    df.groupby("impression_bucket",
               observed=False)
    ["trend_direction"]
    .value_counts(normalize=True)
    .unstack()
    .round(3)
)


print("Signal 2: Search impressions")
print(impression_check)

Signal 2: Search impressions
trend_direction     down   flat    new  stable     up
impression_bucket                                    
Low                0.376  0.147  0.268   0.086  0.123
Medium Low         0.605  0.006  0.018   0.178  0.193
Medium High        0.626  0.001  0.008   0.225  0.141
High               0.562  0.000  0.003   0.307  0.128


Signal verdicts:

1. Staleness:
CONFIRMED

Older pages show more declining trends. Therefore staleness is a useful refresh signal.

2. Search impressions:
MIXED

High impression pages do not always decline, but they represent valuable opportunities because improvements can affect more users.

## Build the ranked queue

The baseline score combines:

60% content staleness

40% search opportunity

The score is ranked from highest to lowest.

Higher score means higher refresh priority.

The CSV output contains:

- content_id
- refresh_score
- reason_code
- action_label

In [4]:
# Create normalized scores

df["staleness_score"] = (
    df["days_since_last_update"] /
    df["days_since_last_update"].max()
)


df["opportunity_score"] = (
    df["impressions_90d"] /
    df["impressions_90d"].max()
)


# Final baseline score

df["refresh_score"] = (
    0.6 * df["staleness_score"]
    +
    0.4 * df["opportunity_score"]
)



# Reason codes

def reason(row):

    if row["staleness_score"] > 0.6 and row["opportunity_score"] > 0.5:
        return "STALE_HIGH_VALUE"

    elif row["staleness_score"] > 0.6:
        return "STALE_LOW_VALUE"

    elif row["opportunity_score"] > 0.5:
        return "FRESH_HIGH_VALUE"

    else:
        return "LOW_PRIORITY"



df["reason_code"] = df.apply(reason, axis=1)



# Action labels

df["action_label"] = np.where(
    df["refresh_score"] >= 0.6,
    "REFRESH",
    "MONITOR"
)



# Ranking

queue = (
    df[
        [
            "content_id",
            "refresh_score",
            "reason_code",
            "action_label"
        ]
    ]
    .sort_values(
        "refresh_score",
        ascending=False
    )
)


# Create output folder

os.makedirs(
    "work/outputs",
    exist_ok=True
)


queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)


queue.head(10)

,content_id,refresh_score,reason_code,action_label
26242,content_55a5b1c46474,0.600027,STALE_LOW_VALUE,REFRESH
29384,content_f6fdf87348f6,0.600002,STALE_LOW_VALUE,REFRESH
4606,content_3f3576c295f5,0.600001,STALE_LOW_VALUE,REFRESH
24216,content_1b4ec72dafd4,0.598393,STALE_LOW_VALUE,MONITOR
18440,content_8d56efff1e71,0.598392,STALE_LOW_VALUE,MONITOR
6653,content_5fe46e04994d,0.567292,FRESH_HIGH_VALUE,MONITOR
6962,content_f01216059a6a,0.538914,STALE_LOW_VALUE,MONITOR
8631,content_e2b702f4f92b,0.537289,STALE_LOW_VALUE,MONITOR
15608,content_06e19c6486b0,0.537273,STALE_LOW_VALUE,MONITOR
29400,content_2dba2b1f9536,0.509901,FRESH_HIGH_VALUE,MONITOR


## Top-20 review

The following pages are the highest priority according to the baseline rule.

For each page I review:

- Action recommended
- Reason code
- Confidence
- What could make this recommendation wrong

The rule is only decision-support and requires human review.

In [5]:
top20 = queue.head(20).copy()

top20["confidence_note"] = np.where(
    top20["reason_code"]=="STALE_HIGH_VALUE",
    "High confidence because page is old and valuable",
    "Medium confidence because only one signal is strong"
)


top20["wrong_if"] = (
    "Recommendation may be wrong if traffic is seasonal "
    "or the topic is no longer relevant."
)


top20

,content_id,refresh_score,reason_code,action_label,confidence_note,wrong_if
26242,content_55a5b1c46474,0.600027,STALE_LOW_VALUE,REFRESH,Medium confidence because only one signal is s...,Recommendation may be wrong if traffic is seas...
29384,content_f6fdf87348f6,0.600002,STALE_LOW_VALUE,REFRESH,Medium confidence because only one signal is s...,Recommendation may be wrong if traffic is seas...
4606,content_3f3576c295f5,0.600001,STALE_LOW_VALUE,REFRESH,Medium confidence because only one signal is s...,Recommendation may be wrong if traffic is seas...
24216,content_1b4ec72dafd4,0.598393,STALE_LOW_VALUE,MONITOR,Medium confidence because only one signal is s...,Recommendation may be wrong if traffic is seas...
18440,content_8d56efff1e71,0.598392,STALE_LOW_VALUE,MONITOR,Medium confidence because only one signal is s...,Recommendation may be wrong if traffic is seas...
6653,content_5fe46e04994d,0.567292,FRESH_HIGH_VALUE,MONITOR,Medium confidence because only one signal is s...,Recommendation may be wrong if traffic is seas...
6962,content_f01216059a6a,0.538914,STALE_LOW_VALUE,MONITOR,Medium confidence because only one signal is s...,Recommendation may be wrong if traffic is seas...
8631,content_e2b702f4f92b,0.537289,STALE_LOW_VALUE,MONITOR,Medium confidence because only one signal is s...,Recommendation may be wrong if traffic is seas...
15608,content_06e19c6486b0,0.537273,STALE_LOW_VALUE,MONITOR,Medium confidence because only one signal is s...,Recommendation may be wrong if traffic is seas...
29400,content_2dba2b1f9536,0.509901,FRESH_HIGH_VALUE,MONITOR,Medium confidence because only one signal is s...,Recommendation may be wrong if traffic is seas...


## Weak picks and leakage check

Some recommendations may be incorrect.

Examples:

- A page may be old but still performing well.
- High impressions may not mean improvement is possible.
- Some topics naturally lose interest over time.

The rule does not use future information.

It only uses current page signals.

In [6]:
# Check columns used in scoring

used_features = [
    "days_since_last_update",
    "impressions_90d"
]


print("Features used:")
for x in used_features:
    print("-",x)



print("\nLeakage check completed.")

print(
    "No future traffic, post-refresh results, "
    "or outcome labels were used."
)

Features used:
- days_since_last_update
- impressions_90d

Leakage check completed.
No future traffic, post-refresh results, or outcome labels were used.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.